In [2]:
import os
# Java 8 path
os.environ['JAVA_HOME'] = '/Library/Java/JavaVirtualMachines/temurin-8.jdk/Contents/Home'

# PySpark path
os.environ['SPARK_HOME'] = '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pyspark'

In [3]:
from pyspark.sql import SparkSession
import re 
spark = SparkSession.builder \
    .appName("Hadoop Analysis word Count") \
    .getOrCreate() 
sc = spark.sparkContext

25/12/14 10:19:22 WARN Utils: Your hostname, Dikshantas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.12.9.160 instead (on interface en0)
25/12/14 10:19:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/14 10:19:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
text = """Apache Spark is a unified analytics engine for big data processing. Spark provides high level APIs in 
Java,Scala, python and in R . Spark is designed for fast and easy-to-use machine learning."""

In [6]:
from pyspark.sql.functions import explode,split,lower,col,count 

In [7]:
df = spark.createDataFrame([(text,)],["text"])

In [8]:
df.show()

+--------------------+
|                text|
+--------------------+
|Apache Spark is a...|
+--------------------+



In [10]:
words_df = (df.select(explode(split(lower(col("text")), r"[s\W]+ ")).alias("word")).filter(col("word") != ""))
words_df.show()


+--------------------+
|                word|
+--------------------+
|      apache spark i|
|  a unified analytic|
|engine for big da...|
|       spark provide|
|      high level api|
|     in \njava,scala|
|     python and in r|
|             spark i|
|designed for fast...|
+--------------------+



In [11]:
word_count_df = words_df.groupBy("word").count().orderBy(col("count").desc())
word_count_df.show()

[Stage 9:>                                                          (0 + 8) / 8]

+--------------------+-----+
|                word|count|
+--------------------+-----+
|     in \njava,scala|    1|
|engine for big da...|    1|
|      high level api|    1|
|     python and in r|    1|
|designed for fast...|    1|
|      apache spark i|    1|
|       spark provide|    1|
|             spark i|    1|
|  a unified analytic|    1|
+--------------------+-----+



In [14]:
# Method 2 creation of RDDS (Resillient distributed DataSet)
rdd = sc.parallelize([text])

In [15]:
words_rdd = rdd.flatMap(lambda line: re.findall(r'\b\w+\b', line.lower()))


In [16]:
pairs_rdd = words_rdd.map(lambda word :(word,1))

In [17]:
words_count_rdd = pairs_rdd.reduceByKey(lambda a,b:a+b)

In [24]:
sorted_counts = words_count_rdd.sortBy(lambda x: x[1] , ascending = False)

In [26]:
for word, count in sorted_counts.collect():
    print(f"{word} : {count}")
    

spark : 3
for : 2
in : 2
and : 2
is : 2
engine : 1
python : 1
r : 1
machine : 1
learning : 1
a : 1
processing : 1
unified : 1
high : 1
scala : 1
designed : 1
use : 1
apache : 1
data : 1
level : 1
fast : 1
analytics : 1
provides : 1
java : 1
to : 1
big : 1
apis : 1
easy : 1


In [27]:
# Rdd - Transofrmation function - mapping function -> unless we call it 
# Action function 